Estimation de l'effet causal de la fermeture des établissement 'social' sur le vote RN à l'aide d'un staggered DID

Import des bibliothèques

In [1]:
pip install csa-py polars

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement csa-py (from versions: none)
ERROR: No matching distribution found for csa-py


In [3]:
import csa-py as csa
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

SyntaxError: invalid syntax (2858975813.py, line 1)

Import des données

In [ ]:
# Social
df_rnp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rnp.csv", index_col=0)
df_rnp['codecommune'] = df_rnp['codecommune'].astype(str).str.zfill(5)

df_rp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rp.csv", index_col=0)
df_rp['codecommune'] = df_rp['codecommune'].astype(str).str.zfill(5)

df_ui = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_ui.csv", index_col=0)
df_ui['codecommune'] = df_ui['codecommune'].astype(str).str.zfill(5)

df_ud = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_d.csv", index_col=0)
df_ud['codecommune'] = df_ud['codecommune'].astype(str).str.zfill(5)

C:\Users\yancr\AppData\Local\Temp\ipykernel_12448\2641705729.py:2: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rnp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rnp.csv", index_col=0)
C:\Users\yancr\AppData\Local\Temp\ipykernel_12448\2641705729.py:5: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rp = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_rp.csv", index_col=0)
C:\Users\yancr\AppData\Local\Temp\ipykernel_12448\2641705729.py:8: DtypeWarning: Columns (0: Annee) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ui = pd.read_csv(r"C:\Users\yancr\Documents\ENSAE_V2\STATAPP\V2\STATAPP_V2\Données\Partie Econometrie\communes_social_ui.csv", index_col=0)


Fonction d'attribution du traitement

In [ ]:
def traitement(df):
    # 1. On définit notre seuil de traitement à partir de l'année 2002
    df_2002 = df[df['Annee'] == 1998].copy()
    df_2002['seuil'] = df_2002['total_equipements'] * 0.6

    # 2. On renseigne ce seuil dans le DataFrame pour comparer les niveaux annuels
    mapping_seuil = df_2002.set_index('codecommune')['seuil']
    df['seuil'] = df['codecommune'].map(mapping_seuil)

    # 3. Première passe : condition simple (1 si on passe sous le seuil, 0 sinon)
    df['traitée'] = (df['total_equipements'] < df['seuil']).astype(int)

    # 4. ÉTAPE CRUCIALE : Trier le tableau pour garantir l'ordre chronologique
    df = df.sort_values(by=['codecommune', 'Annee'])

    # 5. L'EFFET CLIQUET : On applique un maximum cumulé par commune.
    # Dès qu'une commune obtient un 1, toutes les lignes suivantes pour cette commune vaudront 1.
    df['traitée'] = df.groupby('codecommune')['traitée'].cummax()

    return df

## Calcul du traitement

### RNP

In [ ]:
df_rnp = traitement(df_rnp)

### RP

In [ ]:
df_rp = traitement(df_rp)

### UI

In [ ]:
df_ui = traitement(df_ui)

### UD

In [ ]:
df_ud = traitement(df_ud)

## DID

Definition des cohortes

In [ ]:
# Supposons que ton dataframe est df_rnp et qu'il ressemble à ça :
# colonnes : ['code_commune', 'annee', 'traitement', 'vote_RN_pres', ...]

def preparer_cohorte_cs(df, id_col='code_commune', time_col='Annee', treat_col='traitée'):
    """
    Ajoute la colonne 'first_treat_year' nécessaire pour Callaway & Sant'Anna.
    Les communes jamais traitées recevront la valeur 0.
    """
    df_mod = df.copy()
    
    # 1. Isoler les lignes où les communes sont effectivement traitées
    df_traitees = df_mod[df_mod[treat_col] == 1]
    
    # 2. Trouver la PREMIÈRE année de traitement pour chaque commune (le fameux F_g)
    premiere_annee = df_traitees.groupby(id_col)[time_col].min().reset_index()
    
    # Renommer la colonne pour la préparation au merge
    premiere_annee.rename(columns={time_col: 'first_treat_year'}, inplace=True)
    
    # 3. Fusionner cette information avec le DataFrame complet
    df_mod = df_mod.merge(premiere_annee, on=id_col, how='left')
    
    # 4. Gérer le groupe de contrôle (les "never treated")
    # Dans la logique de Callaway & Sant'Anna / pydid, on assigne l'année 0 à ceux qui ne sont jamais traités.
    df_mod['first_treat_year'] = df_mod['first_treat_year'].fillna(0).astype(int)
    
    return df_mod

Definition du modele

In [ ]:
def cs_did_event_study_csa(df, election='pres', party='RN', 
                           id_col='code_commune', time_col='annee', 
                           first_treat_col='first_treat_year'):
    """
    Implémente l'estimateur de Callaway & Sant'Anna (2021) via le vrai package `csa-py`.
    """
    # 1. Nom de la variable dépendante
    outcome_col = f"vote_{party}_{election}"
    
    if outcome_col not in df.columns:
        raise ValueError(f"La colonne {outcome_col} est introuvable dans le DataFrame.")
        
    print(f"--- Modèle Callaway & Sant'Anna (via csa-py) ---")
    print(f"Outcome : {outcome_col}")
    
    # 2. Nettoyage : Callaway & Sant'Anna nécessite que les "Never treated" soient à 0
    df_cs = df.copy()
    df_cs[first_treat_col] = df_cs[first_treat_col].fillna(0).astype(int)
    df_cs = df_cs.dropna(subset=[outcome_col, first_treat_col, id_col, time_col])
    
    # csa-py fonctionne avec la librairie Polars (plus rapide que Pandas)
    df_pl = pl.DataFrame(df_cs)
    
    # 3. Étape 1 : Estimation des Group-Time ATT (ATT(g,t))
    # method="dr" utilise l'estimateur Doubly Robust (recommandé par C&S)
    # control="never" compare les communes traitées à celles qui ne le sont JAMAIS
    res = csa.estimate(
        data=df_pl,
        outcome=outcome_col,
        unit=id_col,
        group=first_treat_col,
        time=time_col,
        control="never",
        method="dr", 
        verbose=True
    )
    
    # 4. Étape 2 : Agrégation en Event-Study (Effets dynamiques relatifs à l'année de traitement)
    dynamic_te = csa.agg_te(res, method="dynamic")
    
    print("\nRésultats de l'Event-Study (Effets Dynamiques) :")
    print(dynamic_te.estimates)
    
    return res, dynamic_te

### RNP

In [ ]:
# ==========================================
# EXÉCUTION
# ==========================================
# On applique la fonction à ton DataFrame
df_rnp = preparer_cohorte_cs(df_rnp, 
                             id_col='codecommune', 
                             time_col='Annee', 
                             treat_col='traitée') # Remplace par le vrai nom de ta colonne traitement

# Vérification rapide du résultat
print("Répartition des cohortes de traitement :")
print(df_rnp[['codecommune', 'first_treat_year']].drop_duplicates()['first_treat_year'].value_counts().sort_index())

In [ ]:
# ==========================================
# EXEMPLE D'UTILISATION ET TRACÉ DU GRAPHIQUE
# ==========================================

# 1. On lance l'estimation
res_gt, es_results = cs_did_event_study_csa(df_rnp, election='pres', party='RN')

# 2. On récupère le tableau des résultats (conversion Polars -> Pandas)
df_plot = es_results.estimates.to_pandas()

# 3. Tracé de l'Event Study (Ajuste les noms de colonnes 'time', 'att', 'se' selon la sortie de csa-py)
plt.figure(figsize=(10, 6))
plt.errorbar(df_plot['time'], df_plot['att'], yerr=1.96 * df_plot['se'], 
            fmt='o', color='b', capsize=5, label='ATT(e) avec IC 95%')
plt.axhline(0, color='red', linestyle='--')
plt.axvline(0, color='grey', linestyle=':', label='Année du traitement')
plt.title(f"Callaway & Sant'Anna : Effet de la fermeture sur le vote RN (Présidentielle)")
plt.xlabel("Élections relatives à la fermeture (Event Time)")
plt.ylabel("Effet sur le vote (points de %)")
plt.legend()
plt.show()

--- Modèle Callaway & Sant'Anna ---
Outcome : vote_RN_pres


AttributeError: module 'pydid' has no attribute 'Pydid'

### RP

### UI

### UD

In [ ]:
import scipy.stats as st

# Les valeurs de votre test Placebo
estimate_placebo = -0.005310 
se_placebo =  0.001607 

# 1. Calcul du Z-score
z_score = estimate_placebo / se_placebo

# 2. Calcul de la p-value bilatérale
p_value = 2 * (1 - st.norm.cdf(abs(z_score)))

print(f"Z-score : {z_score:.3f}")
print(f"P-value du Placebo : {p_value:.3f}")

Z-score : -3.304
P-value du Placebo : 0.001
